# Toxic Comment Detection — Assess Severity

**Pipeline stage**: `assess-severity`  
**Upstream**: `run-inference` → produces `run_inference_output.json`  
**Downstream**: `recommend-moderation-action` → receives `assess_severity_output.json`

This notebook converts the raw inference output into a business-facing severity assessment that can be shown to moderators or consumed by downstream policy skills.

## 1. Setup Paths and Imports

In [ ]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Any

from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()

def detect_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    markers = ["run_inference_output.json", "select_model_output.json", "raw_data"]
    for candidate in candidates:
        if sum((candidate / marker).exists() for marker in markers) >= 2:
            return candidate
    raise FileNotFoundError("Could not automatically detect project root from the current notebook location.")

PROJECT_ROOT = detect_project_root(NOTEBOOK_DIR)
RUN_INFERENCE_OUTPUT_PATH = PROJECT_ROOT / "run_inference_output.json"
OUTPUT_PATH = PROJECT_ROOT / "assess_severity_output.json"

print(f"Project root                : {PROJECT_ROOT}")
print(f"Run inference state path    : {RUN_INFERENCE_OUTPUT_PATH}")
print(f"Assess severity output path : {OUTPUT_PATH}")

## 2. Helper Functions

In [ ]:
def load_json(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Required JSON not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def assess_severity(inference_state: Dict[str, Any]) -> Dict[str, Any]:
    comment_text = inference_state["comment_text"]
    toxicity_probability = float(inference_state["toxicity_probability"])
    confidence = float(inference_state["confidence"])
    is_toxic = bool(inference_state["is_toxic"])
    predicted_label = inference_state["predicted_label"]
    threshold_used = float(inference_state["threshold_used"])

    if not is_toxic:
        if toxicity_probability < 0.2:
            severity_label = "clean"
            severity_rank = 0
            escalation_required = False
            review_priority = "none"
            explanation = "The comment is well below the toxic threshold and does not show a meaningful moderation risk."
        else:
            severity_label = "borderline"
            severity_rank = 1
            escalation_required = False
            review_priority = "low"
            explanation = "The comment is currently classified as non-toxic, but it is close enough to the decision boundary that downstream review logic should treat it as borderline."
    else:
        if toxicity_probability >= 0.9 and confidence >= 0.8:
            severity_label = "critical"
            severity_rank = 4
            escalation_required = True
            review_priority = "urgent"
            explanation = "The comment is classified as toxic with both very high probability and high confidence, suggesting a severe moderation risk that should be escalated immediately."
        elif toxicity_probability >= 0.75:
            severity_label = "high"
            severity_rank = 3
            escalation_required = True
            review_priority = "high"
            explanation = "The comment is classified as toxic with strong probability, indicating a material risk of abuse or harassment and a likely need for moderator intervention."
        elif toxicity_probability >= 0.6:
            severity_label = "medium"
            severity_rank = 2
            escalation_required = False
            review_priority = "medium"
            explanation = "The comment is classified as toxic, but the model signal is moderate rather than overwhelming. It likely warrants review or a softer intervention."
        else:
            severity_label = "low"
            severity_rank = 1
            escalation_required = False
            review_priority = "low"
            explanation = "The comment crossed the toxicity threshold, but only weakly. It should be monitored or reviewed before a strong moderation action is taken."

    summary_for_moderator = (
        f"Predicted {predicted_label} with toxicity probability {toxicity_probability:.4f} "
        f"and confidence {confidence:.4f}. Assigned severity: {severity_label}."
    )

    return {
        "comment_text": comment_text,
        "selected_model_id": inference_state["selected_model_id"],
        "selected_model_label": inference_state.get("selected_model_label"),
        "predicted_label": predicted_label,
        "predicted_class_id": inference_state["predicted_class_id"],
        "is_toxic": is_toxic,
        "toxicity_probability": round(toxicity_probability, 4),
        "non_toxic_probability": round(float(inference_state["non_toxic_probability"]), 4),
        "confidence": round(confidence, 4),
        "threshold_used": threshold_used,
        "severity_label": severity_label,
        "severity_rank": severity_rank,
        "review_priority": review_priority,
        "escalation_required": escalation_required,
        "severity_explanation": explanation,
        "summary_for_moderator": summary_for_moderator,
        "selection_justification": inference_state.get("selection_justification"),
        "bias_assessment": inference_state.get("bias_assessment"),
        "inference_timestamp_utc": inference_state.get("inference_timestamp_utc"),
        "severity_assessed_at_utc": datetime.now(timezone.utc).isoformat()
    }


## 3. Load Upstream State

In [ ]:
inference_state = load_json(RUN_INFERENCE_OUTPUT_PATH)
display(inference_state)

## 4. Assess Severity

In [ ]:
severity_output = assess_severity(inference_state)
display(severity_output)

## 5. Write Output to Agent State

In [ ]:
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(severity_output, f, indent=4, ensure_ascii=False)

print(f"Saved assess severity output to: {OUTPUT_PATH}")

## 6. Downstream Output Contract

Downstream skills should read these fields from `assess_severity_output.json`:

- `comment_text`: original comment carried through from inference
- `predicted_label`: binary model decision (`toxic` / `non-toxic`)
- `toxicity_probability`: toxic-class probability from the model
- `confidence`: distance from the decision boundary
- `severity_label`: business-facing severity band (`clean`, `borderline`, `low`, `medium`, `high`, `critical`)
- `severity_rank`: numeric severity level for downstream rule logic
- `review_priority`: queue priority for human moderation (`none`, `low`, `medium`, `high`, `urgent`)
- `escalation_required`: whether the case should be escalated for stronger action
- `severity_explanation`: plain-language explanation of the severity decision
- `summary_for_moderator`: one-line summary suitable for UI display
